# AutoVision ResNet50 Training on Google Colab

This notebook runs the same training flow as the local project, using the dataset zip `new_data_train_ai_mix.zip` from Google Drive or direct upload.

Expected dataset layout after extraction:

```text
new_data_train_ai_mix/
  F1/
  HATCHBACK/
  MICRO/
  PICK_UP/
  SEDAN/
  STATION_WAGON/
  SUV/
  VAN/
```

## 1. Runtime Setup

Before running: in Colab, choose `Runtime > Change runtime type > A100 GPU` or another GPU runtime.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

REPO_URL = "https://github.com/Abd2023/AutoVision.git"
PROJECT_ROOT = Path("/content/AutoVision")
DATASET_NAME = "new_data_train_ai_mix"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME
DATASET_FILE_ID = "1DyVXdhQRGbw5DqDL5p2iypU-zFkjzO3E"
DATASET_ZIP_PATH = Path(f"/content/{DATASET_NAME}.zip")
COLAB_BATCH_SIZE = 64
COLAB_NUM_WORKERS = 8

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "resnet50_clean_round4"
ERROR_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "error_analysis" / "resnet50_clean_round4"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATA_ROOT)
print("Dataset zip:", DATASET_ZIP_PATH)
print("Train batch size:", COLAB_BATCH_SIZE)
print("Data loader workers:", COLAB_NUM_WORKERS)

In [ ]:
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
print("Current directory:", Path.cwd())

In [ ]:
%pip install -q timm matplotlib scikit-learn pillow gdown

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA is not available. Change the Colab runtime to GPU before training.")

## 2. Fetch Dataset

Preferred flow:

- download `new_data_train_ai_mix.zip` from your Google Drive share link
- extract it into local Colab storage under `/content`

Fallback flow:

- if the direct Drive download fails, upload `new_data_train_ai_mix.zip` manually

The notebook places the final dataset at `AutoVision/data/new_data_train_ai_mix` and trains from local Colab disk, not from mounted Drive.

In [ ]:
import gdown
from google.colab import files

PROJECT_CLASSES = [
    "F1",
    "HATCHBACK",
    "MICRO",
    "PICK_UP",
    "SEDAN",
    "STATION_WAGON",
    "SUV",
    "VAN",
]

def looks_like_class_root(path: Path) -> bool:
    return all((path / class_name).is_dir() for class_name in PROJECT_CLASSES)

def ensure_dataset_zip() -> Path:
    if DATASET_ZIP_PATH.exists():
        print("Using existing local zip:", DATASET_ZIP_PATH)
        return DATASET_ZIP_PATH

    drive_url = f"https://drive.google.com/uc?id={DATASET_FILE_ID}"
    try:
        print("Downloading dataset zip from Google Drive...")
        gdown.download(drive_url, str(DATASET_ZIP_PATH), quiet=False, fuzzy=True)
    except Exception as exc:
        print(f"Direct Drive download failed: {exc}")

    if DATASET_ZIP_PATH.exists():
        print("Downloaded dataset zip to:", DATASET_ZIP_PATH)
        return DATASET_ZIP_PATH

    print("Upload your dataset zip now. Expected name: new_data_train_ai_mix.zip")
    uploaded = files.upload()
    zip_files = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if not zip_files:
        raise RuntimeError("No zip file available. Provide new_data_train_ai_mix.zip from Drive or upload it manually.")

    uploaded_zip = zip_files[0]
    if DATASET_ZIP_PATH.exists():
        DATASET_ZIP_PATH.unlink()
    shutil.move(str(uploaded_zip), str(DATASET_ZIP_PATH))
    print("Uploaded dataset zip to:", DATASET_ZIP_PATH)
    return DATASET_ZIP_PATH

def install_dataset() -> None:
    if DATA_ROOT.exists() and looks_like_class_root(DATA_ROOT):
        print("Dataset already installed:", DATA_ROOT)
        return

    dataset_zip = ensure_dataset_zip()

    extract_root = Path("/content/uploaded_dataset")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(dataset_zip, "r") as archive:
        archive.extractall(extract_root)

    candidates = [
        extract_root / DATASET_NAME,
        extract_root,
    ]
    candidates.extend(path for path in extract_root.iterdir() if path.is_dir())
    source_root = next((path for path in candidates if looks_like_class_root(path)), None)
    if source_root is None:
        raise RuntimeError("Could not find class folders inside uploaded zip.")

    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    shutil.move(str(source_root), str(DATA_ROOT))
    print("Dataset installed at:", DATA_ROOT)

install_dataset()

In [ ]:
counts = {class_name: len([p for p in (DATA_ROOT / class_name).rglob("*") if p.is_file()]) for class_name in PROJECT_CLASSES}
print(counts)
missing = [class_name for class_name in PROJECT_CLASSES if not (DATA_ROOT / class_name).is_dir()]
if missing:
    raise RuntimeError(f"Missing class folders: {missing}")

## 3. Verify Script Version

This checks that the Colab copy of the repo has the newer script arguments used by your local workflow.

In [ ]:
train_help = subprocess.check_output(["python", "notebooks/train_resnet50.py", "--help"], text=True)
freeze_help = subprocess.check_output(["python", "notebooks/freeze_dataset.py", "--help"], text=True)

required_train_flags = ["--model-name", "--loss-type"]
required_freeze_flags = ["--dedupe-exact"]
missing_train_flags = [flag for flag in required_train_flags if flag not in train_help]
missing_freeze_flags = [flag for flag in required_freeze_flags if flag not in freeze_help]

if missing_train_flags or missing_freeze_flags:
    raise RuntimeError(
        "The cloned GitHub repo is missing required script options. "
        f"Missing train flags: {missing_train_flags}; missing freeze flags: {missing_freeze_flags}. "
        "Push/upload the latest local scripts before running training."
    )

print("Script options verified.")

## 4. Freeze Dataset

This creates `data/processed/train`, `data/processed/val`, and `data/processed/test` from `data/new_data_train_ai_mix`.

In [ ]:
!python notebooks/freeze_dataset.py --clear --raw-root data/new_data_train_ai_mix --background white --image-size 224 --max-f1 1000 --max-per-class 1000 --dedupe-exact

In [ ]:
processed_root = PROJECT_ROOT / "data" / "processed"
processed_counts = {}
for split in ["train", "val", "test"]:
    processed_counts[split] = {
        class_name: len([p for p in (processed_root / split / class_name).glob("*") if p.is_file()])
        for class_name in PROJECT_CLASSES
    }
processed_counts

## 5. Train ResNet50

This keeps the same training logic as your local run, but uses Colab-friendly throughput settings so the GPU is not starved by the dataloader.

In [ ]:
!python notebooks/train_resnet50.py --data-root data/processed --output-dir notebooks/outputs/resnet50_clean_round4 --model-name resnet50 --epochs 25 --freeze-epochs 3 --batch-size {COLAB_BATCH_SIZE} --device cuda --num-workers {COLAB_NUM_WORKERS} --patience 8 --loss-type cross_entropy

## 6. Analyze Test Errors

In [ ]:
!python notebooks/analyze_resnet50_errors.py --data-root data/processed --checkpoint notebooks/outputs/resnet50_clean_round4/best_resnet50.pt --output-dir notebooks/outputs/error_analysis/resnet50_clean_round4 --split test --device cuda

In [ ]:
print("Test metrics:")
!cat notebooks/outputs/resnet50_clean_round4/test_metrics.json
print("\nConfusion counts:")
!cat notebooks/outputs/error_analysis/resnet50_clean_round4/confusion_counts_test.csv

## 7. Download Results

In [ ]:
from google.colab import files

download_zip = Path("/content/resnet50_clean_round4_colab_outputs.zip")
if download_zip.exists():
    download_zip.unlink()

with zipfile.ZipFile(download_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in [OUTPUT_DIR, ERROR_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(PROJECT_ROOT))

print("Created:", download_zip)
files.download(str(download_zip))